# Day 3 — Class Imbalance Handling, Stratified Splitting & Baseline Benchmarking

## Overview & Engineering Context
In industrial predictive maintenance, failure events are inherently rare (~3.4% of total operating cycles). Treating this problem with standard classification workflows and naive metrics leads to critical pitfalls:

1. **The 96.6% Accuracy Trap:** A model predicting 0 ("Normal") 100% of the time achieves 96.6% accuracy while missing every single catastrophic machine breakdown (False Negatives).
2. **ROC-AUC Deception:** The vast sea of True Negatives ($TN = 9,661$) keeps the False Positive Rate ($FPR = \frac{FP}{FP+TN}$) artificially tiny, inflating ROC-AUC into the high 0.90s even when precision is poor.
3. **PR-AUC as the Source of Truth:** Precision-Recall AUC focuses exclusively on minority class performance ($TP, FP, FN$), where a random guess baseline equals positive prevalence (~0.034).
4. **Physics Preservation over SMOTE:** Synthetic oversampling (SMOTE) risks generating physically impossible telemetry pairs. Instead, we preserve real physics and re-weight gradients via `scale_pos_weight`.

---
### Day 3 Objectives
* [x] Ingest domain features and clean column syntax for XGBoost compatibility.
* [x] Execute an **80/20 Stratified Split** (`stratify=y`) and export immutable sets to `data/processed/`.
* [x] Benchmark **DummyClassifier** (exposing the accuracy illusion).
* [x] Benchmark **Unweighted XGBoost** (`scale_pos_weight=1.0`).
* [x] Benchmark **Cost-Sensitive XGBoost** (`scale_pos_weight=28.52`).
* [x] Validate via **5-Fold Stratified Cross-Validation**.
* [x] Plot and save **Precision-Recall Curves** and **Confusion Matrices**.


In [1]:
import sys
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to Python path
sys.path.append('../src')
from features import engineer_features, clean_column_names

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
import xgboost as xgb

# Plot aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

print("Environment initialized: numpy, pandas, scikit-learn, xgboost ready.")


Environment initialized: numpy, pandas, scikit-learn, xgboost ready.


## 1. Data Ingestion & Target Leakage Audit
We load the raw 10,000 telemetry records, compute our physics-grounded features (`temp_diff`, `temp_ratio`, `power_w`, `overstrain`, `overstrain_ratio`), and sanitize feature names to prevent XGBoost DMatrix syntax errors.

We strictly drop:
* Arbitrary IDs: `UDI`, `Product ID`
* Diagnostic Post-Mortem labels: `TWF`, `HDF`, `PWF`, `OSF`, `RNF` (target leakage).


In [2]:
raw_path = '../data/raw/ai4i2020.csv'
df_raw = pd.read_csv(raw_path)
print(f"Raw dataset: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")

# Apply physics transformations
df_featured = engineer_features(df_raw)

# Target and leakage exclusion
target_col = 'Machine failure'
leakage_cols = ['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', target_col]

y = df_featured[target_col].copy()
X = df_featured.drop(columns=leakage_cols).copy()

# Sanitize column names for XGBoost
X = clean_column_names(X)

# Ordinal encoding for machine quality tier ('L': 0, 'M': 1, 'H': 2)
type_mapping = {'L': 0, 'M': 1, 'H': 2}
X['type'] = X['type'].map(type_mapping).astype(int)

print(f"Cleaned feature matrix: {X.shape[1]} features")
print(f"Feature columns: {list(X.columns)}")
print(f"Target distribution: Total failures = {y.sum()} ({y.mean():.4%})")


Raw dataset: 10000 rows, 14 columns
Cleaned feature matrix: 11 features
Feature columns: ['type', 'air_temperature_k', 'process_temperature_k', 'rotational_speed_rpm', 'torque_nm', 'tool_wear_min', 'temp_diff', 'temp_ratio', 'power_w', 'overstrain', 'overstrain_ratio']
Target distribution: Total failures = 339 (3.3900%)


## 2. Stratified Train / Test Split & Data Preservation
Because failure events occur in only ~3.4% of instances, uniform random sampling creates severe variance between splits. 
We use `stratify=y` to ensure both the training (80%) and testing (20%) sets preserve the exact ~3.39% failure ratio.

We then save `train.csv` and `test.csv` to `data/processed/` to guarantee that all downstream stages (hyperparameter tuning on Day 4, threshold calibration on Day 5, and API testing on Day 6) evaluate against identical data.


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"Training set: {X_train.shape[0]} samples | Failures: {y_train.sum()} ({y_train.mean():.4%})")
print(f"Testing set:  {X_test.shape[0]} samples | Failures: {y_test.sum()} ({y_test.mean():.4%})")

# Persist processed splits
processed_dir = '../data/processed'
os.makedirs(processed_dir, exist_ok=True)

train_export = pd.concat([X_train, y_train], axis=1)
test_export = pd.concat([X_test, y_test], axis=1)

train_export.to_csv(os.path.join(processed_dir, 'train.csv'), index=False)
test_export.to_csv(os.path.join(processed_dir, 'test.csv'), index=False)
print(f"Persisted train.csv ({train_export.shape}) and test.csv ({test_export.shape}) to {processed_dir}")


Training set: 8000 samples | Failures: 271 (3.3875%)
Testing set:  2000 samples | Failures: 68 (3.4000%)
Persisted train.csv ((8000, 12)) and test.csv ((2000, 12)) to ../data/processed


## 3. Benchmarking Baseline 1: The Dummy Classifier (The Accuracy Illusion)
We instantiate a `DummyClassifier(strategy='most_frequent')` that predicts 0 for all instances.
This serves as our sanity benchmark to expose why **Accuracy is deceptive** and why **PR-AUC tells the truth**.


In [4]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)
y_prob_dummy = dummy.predict_proba(X_test)[:, 1]

dummy_acc = accuracy_score(y_test, y_pred_dummy)
dummy_rec = recall_score(y_test, y_pred_dummy, zero_division=0)
dummy_prec = precision_score(y_test, y_pred_dummy, zero_division=0)
dummy_f1 = f1_score(y_test, y_pred_dummy, zero_division=0)
dummy_roc = roc_auc_score(y_test, y_prob_dummy)
dummy_pr = average_precision_score(y_test, y_prob_dummy)

print("="*50)
print("BASELINE 1: DUMMY CLASSIFIER (PREDICT ALWAYS 0)")
print("="*50)
print(f"Accuracy:  {dummy_acc:.4f}  (Deceptive! Appears 96.6% accurate)")
print(f"Recall:    {dummy_rec:.4f}  (Catastrophic! Caught 0% of failures)")
print(f"Precision: {dummy_prec:.4f}")
print(f"F1-Score:  {dummy_f1:.4f}")
print(f"ROC-AUC:   {dummy_roc:.4f}  (Random chance benchmark: 0.50)")
print(f"PR-AUC:    {dummy_pr:.4f}  (Exact positive class prevalence: {y_test.mean():.4f})")
print("="*50)


BASELINE 1: DUMMY CLASSIFIER (PREDICT ALWAYS 0)
Accuracy:  0.9660  (Deceptive! Appears 96.6% accurate)
Recall:    0.0000  (Catastrophic! Caught 0% of failures)
Precision: 0.0000
F1-Score:  0.0000
ROC-AUC:   0.5000  (Random chance benchmark: 0.50)
PR-AUC:    0.0340  (Exact positive class prevalence: 0.0340)


## 4. Benchmarking Baseline 2: Default Unweighted XGBoost
Next, we train a standard `xgb.XGBClassifier` with default parameters and `scale_pos_weight=1.0`.
This reflects what a standard ML workflow outputs without class imbalance awareness.


In [5]:
xgb_unweighted = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42,
    eval_metric='logloss'
)

xgb_unweighted.fit(X_train, y_train)

y_pred_unw = xgb_unweighted.predict(X_test)
y_prob_unw = xgb_unweighted.predict_proba(X_test)[:, 1]

unw_acc = accuracy_score(y_test, y_pred_unw)
unw_rec = recall_score(y_test, y_pred_unw)
unw_prec = precision_score(y_test, y_pred_unw)
unw_f1 = f1_score(y_test, y_pred_unw)
unw_roc = roc_auc_score(y_test, y_prob_unw)
unw_pr = average_precision_score(y_test, y_prob_unw)

print("="*50)
print("BASELINE 2: UNWEIGHTED XGBOOST (scale_pos_weight=1.0)")
print("="*50)
print(f"Accuracy:  {unw_acc:.4f}")
print(f"Recall:    {unw_rec:.4%}  ({int(unw_rec * y_test.sum())}/{y_test.sum()} failures caught)")
print(f"Precision: {unw_prec:.4%}")
print(f"F1-Score:  {unw_f1:.4f}")
print(f"ROC-AUC:   {unw_roc:.4f}")
print(f"PR-AUC:    {unw_pr:.4f}")
print(f"Missed Failures (FN): {int((1 - unw_rec) * y_test.sum())}")
print("="*50)


BASELINE 2: UNWEIGHTED XGBOOST (scale_pos_weight=1.0)
Accuracy:  0.9900
Recall:    79.4118%  (54/68 failures caught)
Precision: 90.0000%
F1-Score:  0.8438
ROC-AUC:   0.9796
PR-AUC:    0.8765
Missed Failures (FN): 14


## 5. Benchmarking Baseline 3: Cost-Sensitive XGBoost (`scale_pos_weight`)
We compute the inverse class frequency on the training set:
$$\text{scale\_pos\_weight} = \frac{N_{\text{neg}}}{N_{\text{pos}}} = \frac{8000 - 271}{271} \approx 28.52$$

This scales the gradient of the positive class by $\approx 28.52\times$, penalizing missed failures during tree split calculations.


In [6]:
scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
print(f"Calculated scale_pos_weight: {scale_pos_weight:.2f}")

xgb_weighted = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)

xgb_weighted.fit(X_train, y_train)

y_pred_w = xgb_weighted.predict(X_test)
y_prob_w = xgb_weighted.predict_proba(X_test)[:, 1]

w_acc = accuracy_score(y_test, y_pred_w)
w_rec = recall_score(y_test, y_pred_w)
w_prec = precision_score(y_test, y_pred_w)
w_f1 = f1_score(y_test, y_pred_w)
w_roc = roc_auc_score(y_test, y_prob_w)
w_pr = average_precision_score(y_test, y_prob_w)

print("="*50)
print(f"BASELINE 3: WEIGHTED XGBOOST (scale_pos_weight={scale_pos_weight:.2f})")
print("="*50)
print(f"Accuracy:  {w_acc:.4f}")
print(f"Recall:    {w_rec:.4%}  ({int(w_rec * y_test.sum())}/{y_test.sum()} failures caught)")
print(f"Precision: {w_prec:.4%}")
print(f"F1-Score:  {w_f1:.4f}")
print(f"ROC-AUC:   {w_roc:.4f}")
print(f"PR-AUC:    {w_pr:.4f}")
print(f"Missed Failures (FN): {int((1 - w_rec) * y_test.sum())}")
print("="*50)


Calculated scale_pos_weight: 28.52
BASELINE 3: WEIGHTED XGBOOST (scale_pos_weight=28.52)
Accuracy:  0.9890
Recall:    82.3529%  (56/68 failures caught)
Precision: 84.8485%
F1-Score:  0.8358
ROC-AUC:   0.9779
PR-AUC:    0.8659
Missed Failures (FN): 12


## 6. Consolidated Baseline Comparison Table
We compare all three models side-by-side across all operational and statistical metrics.


In [7]:
comparison_df = pd.DataFrame({
    'Model': ['Dummy (Most Frequent)', 'XGBoost (Unweighted)', f'XGBoost (Weighted spw={scale_pos_weight:.1f})'],
    'Accuracy': [dummy_acc, unw_acc, w_acc],
    'Recall (Detection)': [dummy_rec, unw_rec, w_rec],
    'Precision (Reliability)': [dummy_prec, unw_prec, w_prec],
    'F1-Score': [dummy_f1, unw_f1, w_f1],
    'ROC-AUC': [dummy_roc, unw_roc, w_roc],
    'PR-AUC': [dummy_pr, unw_pr, w_pr],
    'Missed Failures (FN)': [
        int((1 - dummy_rec) * y_test.sum()),
        int((1 - unw_rec) * y_test.sum()),
        int((1 - w_rec) * y_test.sum())
    ]
})

print(comparison_df.to_string(index=False))


Model  Accuracy  Recall (Detection)  Precision (Reliability)  F1-Score  ROC-AUC   PR-AUC  Missed Failures (FN)
    Dummy (Most Frequent)    0.9660            0.000000                 0.000000    0.0000   0.5000 0.034000                    68
     XGBoost (Unweighted)    0.9900            0.794118                 0.900000    0.8438   0.9796 0.876527                    14
XGBoost (Weighted spw=28.5)  0.9890            0.823529                 0.848485    0.8358   0.9779 0.865882                    12


## 7. 5-Fold Stratified Cross-Validation
To confirm our results are not dependent on a lucky train/test split, we evaluate both models across 5 stratified folds on the training dataset.


In [8]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_pr_unw, cv_pr_w = [], []
cv_rec_unw, cv_rec_w = [], []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, y_tr = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    fold_spw = (len(y_tr) - y_tr.sum()) / y_tr.sum()
    
    m_u = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42, eval_metric='logloss')
    m_w = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, scale_pos_weight=fold_spw, random_state=42, eval_metric='logloss')
    
    m_u.fit(X_tr, y_tr)
    m_w.fit(X_tr, y_tr)
    
    prob_u = m_u.predict_proba(X_va)[:, 1]
    prob_w = m_w.predict_proba(X_va)[:, 1]
    
    cv_pr_unw.append(average_precision_score(y_va, prob_u))
    cv_pr_w.append(average_precision_score(y_va, prob_w))
    cv_rec_unw.append(recall_score(y_va, m_u.predict(X_va)))
    cv_rec_w.append(recall_score(y_va, m_w.predict(X_va)))

print("="*50)
print("5-FOLD STRATIFIED CROSS-VALIDATION SUMMARY")
print("="*50)
print(f"Unweighted Model PR-AUC: {np.mean(cv_pr_unw):.4f} +/- {np.std(cv_pr_unw):.4f}")
print(f"Weighted Model PR-AUC:   {np.mean(cv_pr_w):.4f} +/- {np.std(cv_pr_w):.4f}")
print(f"Unweighted Model Recall: {np.mean(cv_rec_unw):.4%} +/- {np.std(cv_rec_unw):.4%}")
print(f"Weighted Model Recall:   {np.mean(cv_rec_w):.4%} +/- {np.std(cv_rec_w):.4%}")
print("="*50)


5-FOLD STRATIFIED CROSS-VALIDATION SUMMARY
Unweighted Model PR-AUC: 0.8616 +/- 0.0261
Weighted Model PR-AUC:   0.8673 +/- 0.0190
Unweighted Model Recall: 79.3522% +/- 2.0833%
Weighted Model Recall:   82.0000% +/- 4.1082%


## 8. Visualizing Precision-Recall Curves & Confusion Matrices
We generate publication-grade figures comparing:
1. **Precision-Recall Curve:** Demonstrating model precision across all recall operating thresholds.
2. **ROC Curve:** Demonstrating how large True Negative counts mask false alarms.
3. **Confusion Matrices:** Explicitly contrasting unpredicted failures ($FN$).


In [9]:
prec_unw, rec_unw, _ = precision_recall_curve(y_test, y_prob_unw)
prec_w, rec_w, _ = precision_recall_curve(y_test, y_prob_w)
fpr_unw, tpr_unw, _ = roc_curve(y_test, y_prob_unw)
fpr_w, tpr_w, _ = roc_curve(y_test, y_prob_w)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Subplot 1: PR Curve
ax1.plot(rec_unw, prec_unw, color='#1f77b4', lw=2.5, label=f'Unweighted XGB (PR-AUC = {unw_pr:.4f})')
ax1.plot(rec_w, prec_w, color='#d62728', lw=2.5, label=f'Weighted XGB (PR-AUC = {w_pr:.4f})')
ax1.axhline(y=y_test.mean(), color='gray', linestyle='--', lw=1.5, label=f'No-Skill Baseline ({y_test.mean():.4f})')
ax1.set_xlabel('Recall (Detection Rate)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Precision (True Positive Reliability)', fontsize=12, fontweight='bold')
ax1.set_title('Precision-Recall Curves (Test Set N=2,000)', fontsize=13, fontweight='bold')
ax1.set_xlim([0.0, 1.02])
ax1.set_ylim([0.0, 1.05])
ax1.legend(loc='lower left', frameon=True)
ax1.grid(True, linestyle=':', alpha=0.6)

# Subplot 2: ROC Curve
ax2.plot(fpr_unw, tpr_unw, color='#1f77b4', lw=2.5, label=f'Unweighted XGB (ROC-AUC = {unw_roc:.4f})')
ax2.plot(fpr_w, tpr_w, color='#d62728', lw=2.5, label=f'Weighted XGB (ROC-AUC = {w_roc:.4f})')
ax2.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5, label='Random Guess (0.50)')
ax2.set_xlabel('False Positive Rate (FPR)', fontsize=12, fontweight='bold')
ax2.set_ylabel('True Positive Rate (Recall)', fontsize=12, fontweight='bold')
ax2.set_title('ROC Curves (Illustrating Deceptive Scaling)', fontsize=13, fontweight='bold')
ax2.set_xlim([-0.02, 1.02])
ax2.set_ylim([0.0, 1.05])
ax2.legend(loc='lower right', frameon=True)
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
figures_dir = '../reports/figures'
os.makedirs(figures_dir, exist_ok=True)
plt.savefig(os.path.join(figures_dir, 'day3_baseline_pr_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

# Side-by-side Confusion Matrix Display
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

cm_unw = confusion_matrix(y_test, y_pred_unw)
cm_w = confusion_matrix(y_test, y_pred_w)

disp_unw = ConfusionMatrixDisplay(confusion_matrix=cm_unw, display_labels=['Normal', 'Failure'])
disp_unw.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Unweighted XGBoost\nMissed Failures (FN): {cm_unw[1, 0]} / {y_test.sum()}', fontsize=12, fontweight='bold')
axes[0].grid(False)

disp_w = ConfusionMatrixDisplay(confusion_matrix=cm_w, display_labels=['Normal', 'Failure'])
disp_w.plot(ax=axes[1], cmap='Reds', colorbar=False)
axes[1].set_title(f'Weighted XGBoost (spw={scale_pos_weight:.1f})\nMissed Failures (FN): {cm_w[1, 0]} / {y_test.sum()}', fontsize=12, fontweight='bold')
axes[1].grid(False)

plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'day3_confusion_matrices.png'), dpi=300, bbox_inches='tight')
plt.show()


Saved figures to ../reports/figures


### Day 3 Key Findings & Strategic Conclusions:
1. **The Accuracy Paradox Proven:** The dummy classifier achieved **96.6% accuracy** while missing 100% of failures. Accuracy is completely retired as an operational metric.
2. **PR-AUC as the Production North Star:** Unlike ROC-AUC (which hovered around ~0.979 for both models due to $TN$ scale), PR-AUC clearly reflects real minority-class discrimination (~0.86–0.87 vs 0.034 baseline).
3. **Cost-Sensitive Weighting Effect:** Setting `scale_pos_weight = 28.52` increased default detection recall from **79.4% to 82.4%** on the test set, reducing catastrophic unpredicted failures without introducing synthetic data distortion.
4. **Data Isolation:** Clean train (`data/processed/train.csv`, 8,000 samples) and test (`data/processed/test.csv`, 2,000 samples) splits are locked.
5. **Next Step (Day 4):** Build a formal Scikit-Learn `Pipeline` & `ColumnTransformer`, followed by Bayesian/Optuna or Randomized hyperparameter tuning to optimize PR-AUC and prepare for Day 5 threshold calibration.
